# 01 — Hello CKKS: pierwsze szyfrowanie homomorficzne

> Notebook wprowadzający dla grupy projektowej **„Poufna analiza danych medycznych w HE"**.
> Po przejściu wszystkich komórek będziecie umieli **samodzielnie** zaszyfrować
> wektor danych medycznych i policzyć na nim średnią — w taki sposób, że serwer
> wykonujący obliczenia nie widzi pojedynczych wartości.

## Czego się nauczysz

1. Co to jest **CKKS context** i co dokładnie oznaczają jego parametry.
2. Jak utworzyć pierwszy **ciphertext** i co się dzieje pod spodem (`serialize`, rozmiar w KB).
3. Jakie operacje są tanie (`+`, `× skalar`), a jakie drogie (`× ciphertext`) i dlaczego.
4. Czym jest **batching/SIMD** — czemu jeden ciphertext to wektor 4096 wartości.
5. Jak działa **suma wektora przez rotacje Galois** (serce każdej naszej statystyki).
6. **Średnia szyfrowanej kolumny** medycznej — pełny end-to-end.
7. Co dokładnie widzi serwer w przesłanych bajtach (i dlaczego nic z nich nie wyciągnie).
8. Co to **noise budget**, kiedy ciphertext „się psuje", i dlaczego mediana jest droga.

## Architektura, w którą zmierzamy

```text
Lekarz (zaufany)                      Serwer (niezaufany)
  │                                     │
  │ 1. utwórz kontekst CKKS              │
  │ 2. wygeneruj klucze                  │
  │ 3. zaszyfruj kolumnę                 │
  │ 4. wyślij ciphertext  ─────────►     │
  │                                      │ 5. licz: mean = sum * (1/N)
  │     ◄───── zaszyfrowany wynik        │    (serwer NIE WIE, jakie wartości)
  │ 6. odszyfruj                         │
  │ 7. porównaj z plaintextem (oracle)   │
```

Wszystkie komórki w tym notebooku odpalcie po kolei. Jeśli zatrzymasz się na
jakiejś — zatrzymaj **cały notebook** i przedyskutujcie. Kolejne komórki
polegają na obiektach zdefiniowanych wcześniej.

## Sekcja 1 — Importy i sanity check

Sprawdzamy, że TenSEAL i NumPy się ładują. Jeśli to się wywali — wróćcie do
`README.md` i odpalcie `uv sync`.

In [1]:
import time
from base64 import b16encode

import numpy as np
import tenseal as ts

print(f"TenSEAL: {ts.__version__}")
print(f"NumPy:   {np.__version__}")

rng = np.random.default_rng(seed=42)  # deterministycznie, dla powtarzalności

TenSEAL: 0.3.16
NumPy:   2.0.2


## Sekcja 2 — Pierwszy kontekst CKKS

**Kontekst** to w TenSEAL „opakowanie" trzymające:
- parametry schematu (rozmiar pierścienia, moduły),
- klucze (sekretny, publiczny, ewaluacyjne).

Tworząc go raz, używamy do wszystkich operacji — szyfrowania, obliczeń,
deszyfrowania. Jeden kontekst = jedna „instancja" systemu HE.

### Parametry, które wybieramy

| Parametr | Wartość | Co znaczy |
|---|---|---|
| `poly_modulus_degree` | `8192` | Wielkość pierścienia $R_q = \mathbb{Z}_q[X]/(X^N+1)$. Większe → bezpieczniejsze + więcej slotów, ale wolniejsze. |
| `coeff_mod_bit_sizes` | `[60, 40, 40, 60]` | Bity modułów. Pierwszy + ostatni „specjalne", środkowe = liczba poziomów mnożeń (tu: **2**). |
| `global_scale` | `2**40` | Precyzja kodowania. Większa → mniejszy błąd numeryczny, ale szybciej się wyczerpują poziomy. |
| `galois_keys` | tak | Potrzebne do **rotacji** wektora (suma elementów). |
| `relin_keys` | automatycznie | Potrzebne po każdym mnożeniu ciphertext × ciphertext. |

Dla $N = 8192$ i tych modułów dostajemy **≥ 128 bitów bezpieczeństwa** (standard HES 2018).

In [2]:
POLY_MODULUS_DEGREE = 8192
COEFF_MOD_BIT_SIZES = [60, 40, 40, 60]
SCALE = 2**40

context = ts.context(
    scheme=ts.SCHEME_TYPE.CKKS,
    poly_modulus_degree=POLY_MODULUS_DEGREE,
    coeff_mod_bit_sizes=COEFF_MOD_BIT_SIZES,
)
context.global_scale = SCALE
context.generate_galois_keys()

slot_count = POLY_MODULUS_DEGREE // 2
print(f"Poly modulus degree N = {POLY_MODULUS_DEGREE}")
print(f"Slot count (N/2)      = {slot_count}")
print(f"Global scale          = 2^{int(np.log2(context.global_scale))}")
print(f"Has secret key?       = {context.has_secret_key()}")
print(f"Auto relin?           = {context.auto_relin}")
print(f"Auto rescale?         = {context.auto_rescale}")

Poly modulus degree N = 8192
Slot count (N/2)      = 4096
Global scale          = 2^40
Has secret key?       = True
Auto relin?           = True
Auto rescale?         = True


## Sekcja 3 — Pierwszy ciphertext

Szyfrujemy listę trzech liczb. TenSEAL pakuje ją w ciphertext typu `CKKSVector`,
który operacyjnie zachowuje się jak wektor — tylko że pod spodem są wielomiany
w pierścieniu $R_q$, a nie zwykłe floaty.

Zobaczcie:
- Rozmiar w bajtach (porażająco duży — to cena HE),
- Wynik `print(enc)` (nic informacyjnego — tak ma być),
- Po odszyfrowaniu — **mały błąd numeryczny** (CKKS jest *approximate*).

In [3]:
original = [1.0, 2.0, 3.0]

enc = ts.ckks_vector(context, original)

size_bytes = len(enc.serialize())
print(f"Typ:           {type(enc).__name__}")
print(f"Rozmiar:       {size_bytes:,} B  ({size_bytes/1024:.1f} KB)")
print(f"Oryginał:      {original}")
print(f"Odszyfrowane:  {enc.decrypt()}")

Typ:           CKKSVector
Rozmiar:       334,004 B  (326.2 KB)
Oryginał:      [1.0, 2.0, 3.0]
Odszyfrowane:  [0.999999999201696, 1.9999999997872466, 2.9999999999290403]


In [4]:
# Zmierzmy błąd numeryczny — CKKS to schemat PRZYBLIŻONY
decrypted = np.array(enc.decrypt())
abs_err = np.abs(decrypted - np.array(original)).max()
print(f"Maks. błąd bezwzględny: {abs_err:.3e}")
print(f"(dla scale = 2^40 oczekiwany rząd: ~10^-9 do 10^-7)")

Maks. błąd bezwzględny: 7.983e-10
(dla scale = 2^40 oczekiwany rząd: ~10^-9 do 10^-7)


## Sekcja 4 — Operacje arytmetyczne „w ciemno"

Tutaj dzieje się magia. Wykonujemy operacje matematyczne na zaszyfrowanych
wektorach. Po odszyfrowaniu wynik jest taki, jakbyśmy operowali na
plaintextach.

Trzy klasy operacji w CKKS, w kolejności rosnącego kosztu:

1. **ciphertext + ciphertext / ciphertext + plaintext** — tanie, nie konsumują poziomu.
2. **ciphertext × plaintext** (skalar lub wektor) — średnio drogie, konsumują 1 poziom.
3. **ciphertext × ciphertext** — najdroższe, konsumują 1 poziom + wymagają **relinearyzacji**.

In [5]:
a_plain = [10.0, 20.0, 30.0]
b_plain = [1.0, 2.0, 3.0]

a_enc = ts.ckks_vector(context, a_plain)
b_enc = ts.ckks_vector(context, b_plain)

# (1) dodawanie ciphertext + ciphertext
sum_enc = a_enc + b_enc
print(f"a + b (HE):      {np.round(sum_enc.decrypt(), 6)}")
print(f"a + b (NumPy):   {np.array(a_plain) + np.array(b_plain)}")

a + b (HE):      [11. 22. 33.]
a + b (NumPy):   [11. 22. 33.]


In [6]:
# (2) operacje ze skalarem / plaintextem
scaled_enc = a_enc * 0.5            # ciphertext × skalar (plaintext)
shifted_enc = a_enc + 100.0          # ciphertext + skalar

print(f"a * 0.5 (HE):    {np.round(scaled_enc.decrypt(), 6)}")
print(f"a + 100 (HE):    {np.round(shifted_enc.decrypt(), 6)}")

a * 0.5 (HE):    [ 5.000001 10.000001 15.000002]
a + 100 (HE):    [110. 120. 130.]


In [7]:
# (3) najdroższe: ciphertext × ciphertext (element-wise)
# Po tej operacji ciphertext „konsumuje poziom".
product_enc = a_enc * b_enc
print(f"a * b (HE):      {np.round(product_enc.decrypt(), 6)}")
print(f"a * b (NumPy):   {np.array(a_plain) * np.array(b_plain)}")

# UWAGA: serwer wykonał ta operację BEZ ZNAJOMOŚCI wartości a, b.
# Widział tylko ciphertexty (losowe bajty). Wynik także jest ciphertextem.

a * b (HE):      [10.000001 40.000005 90.000012]
a * b (NumPy):   [10. 40. 90.]


## Sekcja 5 — Batching = wektor liczb w jednym ciphertekście (SIMD)

**Kluczowe pojęcie CKKS:** jeden ciphertext nie szyfruje pojedynczej liczby,
tylko **wektor długości do `N/2`** (u nas: **4096**). Operacja na ciphertekście
działa **element-wise** na całym wektorze jednocześnie — jak SIMD w procesorze.

Konsekwencja praktyczna: 1000 pomiarów BMI pakujemy w jeden ciphertext.
Dodawanie 1000 wartości to **jedna operacja** w HE, nie 1000.

To dlatego HE *bywa* praktyczne dla statystyk.

In [8]:
# Symulujemy 1000 pomiarów BMI ~ N(25, 5)
n = 1000
bmi = rng.normal(loc=25.0, scale=5.0, size=n).tolist()

t0 = time.perf_counter()
bmi_enc = ts.ckks_vector(context, bmi)
t_encrypt = (time.perf_counter() - t0) * 1000

print(f"Zaszyfrowano {n} wartości w JEDNYM ciphertekście.")
print(f"Rozmiar:    {len(bmi_enc.serialize())/1024:.1f} KB")
print(f"Czas:       {t_encrypt:.1f} ms")
print(f"Pierwsze 5 wartości po odszyfrowaniu: {np.round(bmi_enc.decrypt()[:5], 3)}")
print(f"Pierwsze 5 oryginalnych:               {np.round(bmi[:5], 3)}")

Zaszyfrowano 1000 wartości w JEDNYM ciphertekście.
Rozmiar:    326.3 KB
Czas:       9.5 ms
Pierwsze 5 wartości po odszyfrowaniu: [26.524 19.8   28.752 29.703 15.245]
Pierwsze 5 oryginalnych:               [26.524 19.8   28.752 29.703 15.245]


In [9]:
# Pokaz mocy SIMD: jedno mnożenie skalarne działa na 1000 wartości naraz
t0 = time.perf_counter()
bmi_scaled_enc = bmi_enc * 0.5
t_mul = (time.perf_counter() - t0) * 1000

decrypted = np.array(bmi_scaled_enc.decrypt())
expected = np.array(bmi) * 0.5
max_err = np.abs(decrypted - expected).max()

print(f"Mnożenie 1000 wartości × 0.5 w HE: {t_mul:.2f} ms (JEDNA operacja).")
print(f"Maks. błąd vs NumPy:               {max_err:.3e}")

Mnożenie 1000 wartości × 0.5 w HE: 1.39 ms (JEDNA operacja).
Maks. błąd vs NumPy:               2.744e-06


## Sekcja 6 — Suma wektora przez rotacje Galois

Mając wektor `[x1, x2, x3, x4, ...]` zaszyfrowany w jednym ciphertekście,
chcemy policzyć `x1 + x2 + ... + xn`. Klasyczny problem: ciphertext jest
„zamknięty" — nie wyciągniemy elementów osobno.

**Trick: rotacje + dodawanie warstwowe.**

```text
                    [x1, x2, x3, x4, x5, x6, x7, x8]
  + rot(1)          [x2, x3, x4, x5, x6, x7, x8, x1]
  =                 [x1+x2, x2+x3, x3+x4, ...]

  + rot(2)          [x3+x4, x4+x5, ...]
  =                 [x1+x2+x3+x4, ...]

  + rot(4)          ...
  = pierwszy slot:  x1+x2+...+x8 ✓
```

Potrzeba `log2(n)` rotacji. Dla 1000 wartości — **10 rotacji**.
Każda rotacja wymaga klucza Galois (dlatego `generate_galois_keys()`).

TenSEAL implementuje to za nas jako `.sum()`.

In [10]:
t0 = time.perf_counter()
bmi_sum_enc = bmi_enc.sum()
t_sum = (time.perf_counter() - t0) * 1000

# Wynik jest w pierwszym slocie
he_sum = bmi_sum_enc.decrypt()[0]
np_sum = float(np.sum(bmi))

print(f"Suma w HE:        {he_sum:.6f}")
print(f"Suma w NumPy:     {np_sum:.6f}")
print(f"Błąd względny:    {abs(he_sum - np_sum)/abs(np_sum):.3e}")
print(f"Czas operacji:    {t_sum:.2f} ms (dla {n} wartości w 1 ciphertekście)")

Suma w HE:        24855.542250
Suma w NumPy:     24855.542245
Błąd względny:    1.879e-10
Czas operacji:    168.79 ms (dla 1000 wartości w 1 ciphertekście)


## Sekcja 7 — Pierwsza ŚREDNIA medyczna w HE ⭐

Mając już sumę, średnia to triwialne mnożenie przez skalar `1/N` (gdzie
`N` jest **publiczne** — serwer wie ilu pacjentów dostał).

Tu wykonujemy **pełen end-to-end**:

1. Klient (lekarz) ma 1000 pomiarów BMI.
2. Klient szyfruje wektor.
3. **Serwer** (symulujemy go w tym notebooku — w produkcji to inna maszyna)
   liczy `sum() * (1/N)` na ciphertekście.
4. Klient odszyfrowuje.
5. Porównujemy z plaintextem (oracle = `numpy.mean`).

To jest **identyczny przepływ**, którego użyjemy w pełnym systemie z FastAPI.

In [11]:
# --- KLIENT --------------------------------------------------------
t0 = time.perf_counter()
data_enc = ts.ckks_vector(context, bmi)
t_enc = (time.perf_counter() - t0) * 1000

# --- SERWER (symulacja, ale obliczenie identyczne) -----------------
# Serwer zna tylko: ciphertext + publiczne N.
t0 = time.perf_counter()
mean_enc = data_enc.sum() * (1.0 / n)
t_eval = (time.perf_counter() - t0) * 1000

# --- KLIENT --------------------------------------------------------
t0 = time.perf_counter()
mean_he = mean_enc.decrypt()[0]
t_dec = (time.perf_counter() - t0) * 1000

# --- ORACLE --------------------------------------------------------
t0 = time.perf_counter()
mean_np = float(np.mean(bmi))
t_np = (time.perf_counter() - t0) * 1000

rel_err = abs(mean_he - mean_np) / abs(mean_np)

print("== Średnia BMI (1000 pacjentów) ==")
print(f"  HE:           {mean_he:.6f}")
print(f"  NumPy:        {mean_np:.6f}")
print(f"  Błąd wzgl.:   {rel_err:.3e}")
print()
print("== Czasy ==")
print(f"  Szyfrowanie:  {t_enc:.2f} ms  (klient)")
print(f"  Obliczenie:   {t_eval:.2f} ms  (serwer)")
print(f"  Deszyfracja:  {t_dec:.2f} ms  (klient)")
print(f"  Razem HE:     {t_enc + t_eval + t_dec:.2f} ms")
print(f"  NumPy:        {t_np:.4f} ms")
print(f"  Narzut HE:    ~{(t_enc + t_eval + t_dec) / max(t_np, 1e-6):.0f}× wolniej")

== Średnia BMI (1000 pacjentów) ==
  HE:           24.855546
  NumPy:        24.855542
  Błąd wzgl.:   1.345e-07

== Czasy ==
  Szyfrowanie:  6.23 ms  (klient)
  Obliczenie:   161.42 ms  (serwer)
  Deszyfracja:  1.37 ms  (klient)
  Razem HE:     169.02 ms
  NumPy:        0.2662 ms
  Narzut HE:    ~635× wolniej


🎉 **Gratulacje!** Właśnie obliczyliście średnią zaszyfrowanej kolumny medycznej.
Wynik jest praktycznie identyczny z plaintextem (błąd ~$10^{-7}$), a serwer
nigdy nie zobaczył pojedynczych wartości.

**Co poszło między klientem a serwerem?** Tylko ciphertexty — losowe bajty,
nieczytelne bez `secret_key`. Zobaczmy to gołym okiem.

## Sekcja 8 — Co serwer widzi?

Z punktu widzenia serwera ciphertext to **strumień losowych bajtów**. Nie da
się z niego nic wnioskować — bezpieczeństwo CKKS opiera się na trudności
problemu **RLWE** (Ring Learning With Errors), który dla naszych parametrów
daje **≥ 128 bitów bezpieczeństwa**.

Zobaczmy pierwsze bajty serializowanego ciphertextu:

In [12]:
raw = data_enc.serialize()
print(f"Ciphertext 1000 wartości BMI:")
print(f"  Pełny rozmiar:        {len(raw):,} B  ({len(raw)/1024:.1f} KB)")
print(f"  Pierwsze 80 bajtów (hex):")
print(f"    {b16encode(raw[:80]).decode()}")
print()
print("Tyle widzi serwer. Bez secret_key nie da się tego odszyfrować nawet znając")
print("poprzednie wartości, kontekst, schemat — RLWE jest udowodnione jako trudne.")

Ciphertext 1000 wartości BMI:
  Pełny rozmiar:        334,137 B  (326.3 KB)
  Pierwsze 80 bajtów (hex):
    0A02E80712A8B2145EA1100401020000281905000000000028B52FFDA0610006002C5C0E8EFD1FDD722D10A0BC6695A7B375DF3A9C07E11678B1B017673530A614729A22ED5F5FEE164649E7C48AB7B2

Tyle widzi serwer. Bez secret_key nie da się tego odszyfrować nawet znając
poprzednie wartości, kontekst, schemat — RLWE jest udowodnione jako trudne.


In [13]:
# WAŻNE: do serwera wysyłamy kontekst BEZ secret_key.
# Domyślnie `serialize()` zachowuje klucze ewaluacji (public, galois, relin)
# i POMIJA secret_key — tak właśnie powinno trafić na serwer.

ctx_for_server = context.serialize(save_secret_key=False)

print(f"Kontekst dla serwera (BEZ secret_key):  {len(ctx_for_server)/1024/1024:.2f} MB")
print()
print("Co serwer ma: public_key + galois_keys + relin_keys")
print("Czego serwer NIE ma: secret_key — bez niego nie odszyfruje niczego.")
print()
print("Klucze ewaluacji (galois, relin) są DUŻE — to one zajmują większość")
print("powyższych megabajtów. Zawierają wiele wersji przesunięć i relinearyzacji,")
print("które umożliwiają serwerowi operowanie na ciphertekstach bez secret_key.")

Kontekst dla serwera (BEZ secret_key):  33.83 MB

Co serwer ma: public_key + galois_keys + relin_keys
Czego serwer NIE ma: secret_key — bez niego nie odszyfruje niczego.

Klucze ewaluacji (galois, relin) są DUŻE — to one zajmują większość
powyższych megabajtów. Zawierają wiele wersji przesunięć i relinearyzacji,
które umożliwiają serwerowi operowanie na ciphertekstach bez secret_key.


## Sekcja 9 — Limity HE: noise budget i kiedy ciphertext się psuje

Każde mnożenie *konsumuje poziom* z `coeff_mod_bit_sizes`. Mając
`[60, 40, 40, 60]` mamy **2 poziomy mnożeń**. Po ich wyczerpaniu wynik staje
się śmieciem (lub TenSEAL rzuca wyjątek).

**Demo:** mnożymy ciphertext sam przez siebie w pętli i patrzymy, co się dzieje
z dokładnością.

In [14]:
# Zaczynamy od czystego ciphertextu zawierającego małe wartości (żeby uniknąć
# overflow po podnoszeniu do kolejnych potęg).
x = 1.05
vec_enc = ts.ckks_vector(context, [x] * 10)

print(f"  start:  x = {x}")
print(f"  -----   {'HE':>14}   {'plaintext':>14}   {'błąd wzgl.':>12}")

expected = x
for i in range(1, 6):
    try:
        vec_enc = vec_enc * vec_enc
        decrypted = vec_enc.decrypt()[0]
        expected = expected * expected
        rel = abs(decrypted - expected) / abs(expected)
        print(f"  iter {i}:  x = {decrypted:14.6f}   {expected:14.6f}   {rel:12.3e}")
    except Exception as e:
        print(f"  iter {i}:  WYJĄTEK — wyczerpano poziomy: {type(e).__name__}")
        break

  start:  x = 1.05
  -----               HE        plaintext     błąd wzgl.
  iter 1:  x =       1.102500         1.102500      1.341e-07
  iter 2:  x =       1.215507         1.215506      9.392e-07
  iter 3:  WYJĄTEK — wyczerpano poziomy: ValueError


Co właśnie zobaczyliście:

- 1. mnożenie → dokładność OK,
- 2. mnożenie → narastający szum, ale wynik jeszcze sensowny,
- 3.+ mnożenie → poziomy wyczerpane: błąd rośnie wykładniczo lub wyjątek.

**Praktyczne wnioski dla naszego projektu:**

| Operacja statystyczna | Głębokość mnożeń | Wystarczy `[60,40,40,60]`? |
|---|---|---|
| Suma, średnia | 0 (tylko dodawania + skalar) | tak |
| Wariancja $E[X^2] - E[X]^2$ | 1 (`x*x`) | tak |
| Std (Newton sqrt, 2 iter) | 3-4 | dodajemy poziomy: `[60,40,40,40,40,60]` |
| **Mediana przez porównania** | $\Theta(\log^2 N)$ | **NIE — dlatego robimy histogram** |

Histogram działa, bo serwer wykonuje tylko **dodawania** (na one-hot wektorach
zaszyfrowanych przez klienta) — głębokość mnożeń = 0.

## Sekcja 10 — Mini-quiz dla grupy

Pięć pytań kontrolnych. Odpowiedzcie sobie nawzajem **bez patrzenia w komórki
kodu** — to dobry sprawdzian rozumienia.

1. **Co się stanie, jeśli zaszyfrujesz wektor dłuższy niż `N/2 = 4096` slotów?**
   *(Podpowiedź: TenSEAL rzuci wyjątek. Trzeba rozbić na wiele ciphertextów —
   ten przypadek obsłużymy w kodzie produkcyjnym.)*

2. **Dlaczego `secret_key` nigdy nie idzie na serwer, a `relin_keys` i
   `galois_keys` tak?**
   *(Podpowiedź: te ostatnie są **kluczami ewaluacji** — pozwalają operować na
   ciphertextach, ale nie da się z nich odzyskać `secret_key`. To są kluczowe
   założenia bezpieczeństwa CKKS.)*

3. **Co dokładnie chroni HE? Wartości danych? Schemat? Czas obliczenia?
   Rozmiar?**
   *(Podpowiedź: tylko **wartości**. Schemat (jakie kolumny), rozmiar (ilu
   pacjentów), czas i wzorzec dostępu są **publiczne** = side channels. To
   ważne dla raportu — pokażcie zrozumienie, co HE NIE chroni.)*

4. **Dlaczego dla wariancji używamy formy rozwiniętej $E[X^2] - E[X]^2$, a
   nie definicyjnej $\frac{1}{N}\sum (x_i - \bar{x})^2$?**
   *(Podpowiedź: forma rozwinięta to **1 mnożenie ciphertext × ciphertext**
   (`x*x`), forma definicyjna wymagałaby `\bar{x}` osobno i potem `(x_i - \bar{x})`
   dla każdego pacjenta — wiele dodatkowych operacji.)*

5. **Gdybyście próbowali policzyć medianę w tym notebooku „uczciwie" przez
   sortowanie w HE, czemu by się nie udało?**
   *(Podpowiedź: sortowanie wymaga porównań, porównanie wymaga aproksymacji
   funkcji `sign(x)` wielomianem stopnia ≥ 7, sieć sortująca to
   $\Theta(N \log^2 N)$ porównań — dla 1000 pacjentów to setki tysięcy
   mnożeń, daleko poza naszym `noise budgetem`. Dlatego robimy histogram.)*

### Co dalej

- Przeczytajcie `docs/theory.md` (jeśli jeszcze nie) — to rozszerzenie tego, co
  widzieliście tutaj.
- Przeczytajcie `docs/architecture.md` — zobaczycie, jak ten sam przepływ
  („szyfruj → wyślij → policz → odeszyfruj") będzie wyglądać z prawdziwym
  serwerem FastAPI.
- **Notebook 02** (`02_stats_demo.ipynb`) — to samo, ale na prawdziwych danych
  medycznych (Synthea + nasz generator PESEL), z wariancją, std i medianą
  przez histogram.
- **Notebook 03** (`03_benchmarks.ipynb`) — wykresy czasu w funkcji liczby
  pacjentów. To te wykresy pójdą do prezentacji.

### Lektury dla ciekawych

- Cheon, Kim, Kim, Song — *Homomorphic Encryption for Arithmetic of Approximate
  Numbers* (CKKS, 2017). [Original paper](https://eprint.iacr.org/2016/421.pdf)
- [Microsoft SEAL manual](https://github.com/microsoft/SEAL) — to leży pod TenSEAL.
- [TenSEAL tutorial notebooks](https://github.com/OpenMined/TenSEAL/tree/main/tutorials)
  — oficjalne, w języku angielskim.